# Intent Classification Using LLM 



**Overview:**

Prompting to an LLM to classify the intention of the user's quety into one of the following:
- Greeting

- Goodbye

- Gratitude

- Asking_mental_health_question

- Out_of_scope




**Step 1 : Build a prompt**

In [34]:
def build_prompt(user_input):
    return f"""
You are an intent classification system.

Classify the user input into ONE of the following labels:
- greeting
- goodbye
- gratitude
- asking_mental_health_question
- out_of_scope

Rules:
- Output ONLY ONE label
- Only return the label.
- Do not explain.
- Be strict.
- Do NOT add punctuation
- Do NOT add extra text
- If message is about coding/programming → out_of_scope
- If message is about math → out_of_scope
- If message is about general knowledge → out_of_scope
- If message contains goodbye → ALWAYS output goodbye unless it is mental health or feelings related putput mental health intent
- If the message contains i guess or i think or maybe and a feeling or emotion → mental health intent
- Mental health intent ONLY if user is asking about feelings, emotions, anxiety, depression, stress

If multiple intents exist, choose the MOST IMPORTANT one:
Priority: asking_mental_health_question > goodbye > greeting > gratitude > out_of_scope

Examples:
Input: "Hello there"
Output: greeting

Input: "Thanks for your help"
Output: gratitude

Input: "I feel anxious all the time"
Output: asking_mental_health_question

Input: "What is 2+2?"
Output: out_of_scope

Now classify:
Input: "{user_input}"
Output:
"""

**Step 2 : Call the LLM**

In [40]:
from groq import Groq
client = Groq(api_key="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")

def classify_intent(user_input):
    prompt = build_prompt(user_input)
    response = client.chat.completions.create(
        model = "llama-3.1-8b-instant",
        messages =[{"role": "user", "content": prompt}],
        temperature = 0
    )
    return response.choices[0].message.content.strip()

Why do we need role?
As it tells the LLM how to interpret each message 

system --> rules 

user --> question 

assistant --> previous model replies , contect tracking 




Temperature ?

Controls randmoness so if the same input is given multiple times , it'll always produce the same output 

0 --> Strict 

1--> Random 

**Step 3 : Safe Layer to avoid going off-format**

In [28]:
VALID_LABELS = {
    "greeting",
    "goodbye",
    "gratitude",
    "asking_mental_health_question",
    "out_of_scope"
}

def safe_classify_intent(user_input):
    intent = classify_intent(user_input).lower()

    if intent not in VALID_LABELS:
        return "out_of_scope"
    
    return intent 

**Step 4 : Test**

In [41]:
test_inputs = [
    "Hi there",
    "I feel really depressed lately",
    "Thanks!!",
    "Can you help me code in Python?",
    "Goodbye",
    "Hi , Bye" ,
    "I am so grateful for your help, but I have to go now. Goodbye!",
    "Hi I feel anxious",
    "Thanks, I’ve been depressed for weeks",
    "Hello, can you help me with my stress?",
    "I don’t know what I’m feeling",
    "I’m fine I guess…",
    "Hey I’ve been feeling stressed",
    "Thanks, I feel like I want to die",
    "I have to go but I feel anxious",
    "How do I train a machine learning model?",
    "What is depression in biology",
    "Hey… I don’t even know… I’m tired of everything",
    "Hi, thanks, bye",
    "Yeah everything is just great…",
    "I love being anxious all the time 🙃",
    "مرحبا أنا مكتئب"
]

for text in test_inputs:
    print(text, "->", safe_classify_intent(text))

Hi there -> greeting
I feel really depressed lately -> asking_mental_health_question
Thanks!! -> gratitude
Can you help me code in Python? -> out_of_scope
Goodbye -> goodbye
Hi , Bye -> goodbye
I am so grateful for your help, but I have to go now. Goodbye! -> goodbye
Hi I feel anxious -> asking_mental_health_question
Thanks, I’ve been depressed for weeks -> asking_mental_health_question
Hello, can you help me with my stress? -> asking_mental_health_question
I don’t know what I’m feeling -> asking_mental_health_question
I’m fine I guess… -> asking_mental_health_question
Hey I’ve been feeling stressed -> asking_mental_health_question
Thanks, I feel like I want to die -> asking_mental_health_question
I have to go but I feel anxious -> asking_mental_health_question
How do I train a machine learning model? -> out_of_scope
What is depression in biology -> out_of_scope
Hey… I don’t even know… I’m tired of everything -> asking_mental_health_question
Hi, thanks, bye -> goodbye
Yeah everything i

**Step 4 : Integrate into RAG**

In [ ]:
# intent = safe_classify_intent(user_input)

# if intent == "greeting":
#     return "Hello! How are you feeling today?"

# elif intent == "goodbye":
#     return "Take care. I'm here if you need support."

# elif intent == "gratitude":
#     return "You're welcome. I'm glad I could help."

# elif intent == "asking_mental_health_question":
#     # → CALL RAG PIPELINE
#     return rag_response(user_input)

# else:
#     return "I'm here to support mental health topics. Could you rephrase?"